In [ ]:
%load_ext autoreload
# autoreload 2：改了 benchmark/ 下的文件不用重启 kernel
%autoreload 2

import os
os.chdir(os.path.expanduser("~/projects/LLM/hw2"))   # benchmark 包以 hw2/ 为根

import torch
from benchmark import attention_sweep, bench_attention, D_MODELS, SEQ_LENS
from cs336_basics.model import scaled_dot_product_attention as sdpa   # §4.1 的被测对象

torch.backends.cuda.matmul.allow_tf32 = False        # fp32 基准：不偷偷走 tensor core
torch.backends.cudnn.allow_tf32 = False

PROF = "profiles"                       # nsys 报告、显存快照（gitignored）
def assets(sec):                        # 每节的落盘目录 notes/assets/s<sec>；各节开头 OUT = assets(n)
    d = f"notes/assets/s{sec}"; os.makedirs(d, exist_ok=True); return d

print(torch.cuda.get_device_name(0), torch.__version__)

## 4 GPU Kernels

PDF §4。编号按 problem 顺序（4.1 `pytorch_attention` = PDF 4.1.1，4.2 `torch_compile` = PDF 4.2，
4.3–4.5 三题在 PDF 里都归在 4.2.2 下）。

In [ ]:
OUT = assets(4)

### 4.1 `pytorch_attention`

被测的是 hw1 的 `scaled_dot_product_attention(q, k, v, mask=None)`，
输入 `[batch, seq, d]` —— batch 固定 8，**去掉 head 维**（题面 (i)）。

每个 (d, seq) 格子量三个数：100 次前向 ms、反向开始前的 `memory_allocated()`、100 次反向 ms；
OOM 的格子记 `status`，不中断 sweep。

In [ ]:
# 冒烟：最小的两格 + steps 调小，验证三个数都出得来、OOM 捕获路径没写错
for d, s in [(16, 256), (16, 1024)]:
    print(bench_attention(sdpa, d, s, steps=5))

In [ ]:
# 全量 4 × 5 = 20 格。大格子预期 OOM —— 那正是题目要的答案
df = attention_sweep(sdpa, f"{OUT}/attention_fp32.md")
df

#### 最小 OOM 配置的显存账

全部 4 个 d 都在 seq=16384 才 OOM → 最小的 OOM 配置是 **d=16, seq=16384**。
逐张量数字节，和上表 seq≤8192 的实测列对账。

In [ ]:
import pandas as pd
MiB = 2**20
B = 8

def account(d, seq):
    qkvo = 4 * B * seq * d * 4            # Q K V O，随 seq 线性
    sp   = 2 * B * seq * seq * 4          # S=QKᵀ/√d 与 P=softmax(S)，随 seq²
    return dict(d=d, seq=seq,
                QKVO_MiB=qkvo/MiB, S_P_MiB=sp/MiB, total_GiB=(qkvo+sp)/2**30)

acc = pd.DataFrame([account(16, s) for s in SEQ_LENS])
acc["实测_GiB"] = list(df[df.d == 16].mem_before_bwd_gib)
acc